## Cell 1 - Setup

In [ ]:
# ── Environment setup: works locally (repo clone) and on Kaggle ─────────────
# Locally: run this notebook from the repo's notebooks/ folder, no edits needed.
# On Kaggle: first run  !git clone https://github.com/arcadehero3-eng/CHENDA_PGSD.git
import os

def _find_repo():
    cands = [os.environ.get("CHENDA_REPO", ""),
             os.path.abspath(".."), os.getcwd(),
             "/kaggle/working/CHENDA_PGSD"]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "src", "chenda_pgsd_v3.py")):
            return os.path.abspath(c)
    raise FileNotFoundError(
        "CHENDA_PGSD repo not found. Run this notebook from the repo's notebooks/ "
        "folder, set CHENDA_REPO, or on Kaggle clone the repo into /kaggle/working.")

ON_KAGGLE   = os.path.isdir("/kaggle/input")
REPO        = _find_repo()
SRC_DIR     = os.path.join(REPO, "src")
BEATS_DIR   = os.path.join(REPO, "audio", "real_recordings")   # segmented beat_N.wav
STRIKES_DIR = os.path.join(REPO, "audio", "raw_strikes")       # centre/edge/hand x 3
WORK        = os.path.join("/kaggle/working" if ON_KAGGLE else os.path.join(REPO, "outputs"), "chenda_pgsd_analysis")
os.makedirs(WORK, exist_ok=True)

def load_pgsd(ns):
    """Load the PGSD model code from src/ into the notebook namespace."""
    path = os.path.join(SRC_DIR, "chenda_pgsd_v3.py")
    src = open(path, encoding="utf-8").read()
    src = src[:src.rfind("\nif __name__")]
    ns["__file__"] = path
    exec(compile(src, path, "exec"), ns)

print(f"Repo   : {REPO}")
print(f"Outputs: {WORK}")


In [1]:
AUDIO_DATASET = BEATS_DIR
MODEL_DATASET = SRC_DIR

import os, json, warnings
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, resample_poly
from scipy.fft import rfft, rfftfreq
from math import gcd
warnings.filterwarnings("ignore")
os.makedirs(f"{WORK}/figures", exist_ok=True)

TARGET_FS = 22050

load_pgsd(globals())

print("Model loaded:")
print(f"  T_REAL={T_REAL}  eta1={ChendaDamping().eta1}  shapes={list(SHAPES.keys())}")
print(f"  pgsd_refine has f1_min/f1_max params:", 'f1_min' in str(pgsd_refine.__code__.co_varnames))

RECORDING_META = {
    "edge_low"   : {"group":"Edge",  "intensity":"Low",  "r_frac":0.90, "skip_ms":40},
    "edge_med"   : {"group":"Edge",  "intensity":"Med",  "r_frac":0.90, "skip_ms":40},
    "edge_high"  : {"group":"Edge",  "intensity":"High", "r_frac":0.90, "skip_ms":40},
    "centre_low" : {"group":"Centre","intensity":"Low",  "r_frac":0.05, "skip_ms":30},
    "centre_med" : {"group":"Centre","intensity":"Med",  "r_frac":0.05, "skip_ms":30},
    "centre_high": {"group":"Centre","intensity":"High", "r_frac":0.05, "skip_ms":30},
    "hand_full"  : {"group":"Hand",  "intensity":"Full", "r_frac":0.50, "skip_ms":30},
    "hand_half"  : {"group":"Hand",  "intensity":"Half", "r_frac":0.50, "skip_ms":30},
    "hand_small" : {"group":"Hand",  "intensity":"Small","r_frac":0.50, "skip_ms":30},
}

audio_files = sorted([
    os.path.join(AUDIO_DATASET, f)
    for f in os.listdir(AUDIO_DATASET)
    if f.lower().endswith(('.wav','.mp3','.flac','.m4a'))
]) if os.path.isdir(AUDIO_DATASET) else []
print(f"Found {len(audio_files)} recordings")

Model loaded:
  T_REAL=4284.0  eta1=1.087  shapes=['Cylinder', 'Hourglass', 'Oval', 'RealChenda']
  pgsd_refine has f1_min/f1_max params: True
Found 50 recordings


In [2]:
def load_audio(path, target_fs=TARGET_FS):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.wav':
        fs, data = wavfile.read(path)
        if   data.dtype == np.int16:  sig = data.astype(float) / 32768.0
        elif data.dtype == np.int32:  sig = data.astype(float) / 2147483648.0
        else:                          sig = data.astype(float)
    else:
        try:
            import soundfile as sf; sig, fs = sf.read(path, always_2d=False)
        except:
            import librosa; sig, fs = librosa.load(path, sr=None, mono=True)
    sig = np.array(sig, dtype=float)
    if sig.ndim == 2: sig = sig.mean(axis=1)
    if fs != target_fs:
        g = gcd(int(target_fs), int(fs))
        sig = resample_poly(sig, target_fs//g, int(fs)//g)
        fs = target_fs
    return sig.astype(np.float64) / (np.abs(sig).max() + 1e-12), int(fs)


def adaptive_window(signal, fs, max_ms=400, noise_factor=3.0):
    
    frame   = max(1, int(fs * 0.010))
    energy  = np.convolve(signal**2, np.ones(frame)/frame, mode='same')
    noise   = np.percentile(energy, 10)          # bottom 10% = noise floor
    thresh  = noise_factor * noise
    above   = np.where(energy > thresh)[0]
    if len(above) < 50:
        return int(fs * max_ms / 1000)           # fallback: full window
    end_samp = min(above[-1] + frame, int(fs * max_ms / 1000))
    return max(end_samp, int(fs * 50 / 1000))    # at least 50ms


def preprocess_v31(path, skip_ms=30, max_ms=400,
                    bp_lo=80, bp_hi=800, threshold_db=-35):
    
    sig_raw, fs = load_audio(path)
    nyq = fs / 2.0
    b, a = butter(4, [bp_lo/nyq, min(bp_hi, nyq*0.97)/nyq], btype='band')
    sig_filt = filtfilt(b, a, sig_raw)
    frame  = max(1, int(fs * 0.005))
    energy = np.convolve(sig_filt**2, np.ones(frame)/frame, mode='same')
    db     = 10 * np.log10(energy + 1e-12)
    thresh = db.max() + threshold_db
    above  = np.where(db > thresh)[0]
    onset  = max(0, above[0]) if len(above) > 0 else 0
    start  = onset + int(fs * skip_ms / 1000)

    nyq  = fs / 2.0
    b, a = butter(4, [bp_lo/nyq, min(bp_hi, nyq*0.97)/nyq], btype='band')
    sig_bp = filtfilt(b, a, sig_raw)

    # Adaptive window on the bandpassed signal after the skip
    sig_from_start = sig_bp[start:]
    win_samps = adaptive_window(sig_from_start, fs, max_ms=max_ms)
    seg = sig_from_start[:win_samps]

    # Pad if too short
    # Pad if too short
    if len(seg) < int(fs * 50 / 1000):
        seg = np.pad(seg, (0, int(fs*50/1000) - len(seg)))
    seg = seg - np.mean(seg)
    seg /= (np.abs(seg).max() + 1e-12)
    snr_db = float(20 * np.log10(
        np.sqrt(np.mean(sig_bp[start:start+int(fs*0.05)]**2) + 1e-12) /
        (np.sqrt(np.mean(sig_bp[-int(fs*0.3):]**2)) + 1e-12) + 1e-12))

    return seg.astype(np.float64), fs, onset, win_samps / fs * 1000, snr_db


def pgsd_decompose_ar(signal, Phi, rel_lam=1e-6):
    #Active-region R2: computed only where |signal| > 5% of peak.#
    G   = Phi.T @ Phi
    lam = rel_lam * np.linalg.norm(G, 'fro')
    c   = np.linalg.solve(G + lam*np.eye(G.shape[0]), Phi.T @ signal)
    fit = Phi @ c
    res = signal - fit
    active = np.abs(signal) > 0.05 * np.abs(signal).max()
    if active.sum() < 50: active = np.ones(len(signal), dtype=bool)
    ss_res = np.sum(res[active]**2)
    ss_tot = np.sum((signal[active] - signal[active].mean())**2) + 1e-12
    return c, res, fit, float(1.0 - ss_res/ss_tot)


print("Pre-processor v3.1 ready.")


Pre-processor v3.1 ready.


## Cell 3 - Build calibrated bases at T=4284 N/m

In [3]:
N_MODES     = 30
SHAPES_LIST = ['Cylinder', 'Hourglass', 'Oval', 'RealChenda']
T_ANALYSIS  = T_REAL

print(f"Building {N_MODES}-mode bases at T={T_ANALYSIS} N/m...")
BASES = {}
for shape in SHAPES_LIST:
    fr, mo, al, r, th, _ = get_spectral_basis(shape, T=T_ANALYSIS, n_modes=N_MODES)
    BASES[shape] = dict(freqs=fr, modes=mo, alphas=al, r=r, th=th)
    T60 = ChendaDamping().T60(2*np.pi*fr[0])
    print(f"  {shape:12s}: f1={fr[0]:.1f}Hz  T60={T60:.3f}s")

REF_F1    = {s: BASES[s]['freqs'][0] for s in SHAPES_LIST}
T60_REF_D = {s: ChendaDamping().T60(2*np.pi*BASES[s]['freqs'][0]) for s in SHAPES_LIST}
print("Done.")

Building 30-mode bases at T=4284.0 N/m...
  Cylinder    : f1=141.4Hz  T60=1.719s
  Hourglass   : f1=222.6Hz  T60=0.755s
  Oval        : f1=131.9Hz  T60=1.937s
  RealChenda  : f1=138.9Hz  T60=1.774s
Done.


## Cell 4 - Run all four fixes on every recording

In [4]:
# v3 results for comparison
V3_R2 = {
    "centre_high":0.2452, "centre_low":0.6904, "centre_med":0.0448,
    "edge_high":0.3057,   "edge_low":0.2282,   "edge_med":0.0064,
    "hand_full":0.7315,   "hand_half":0.5922,  "hand_small":0.6720,
}

R2_RELIABLE_THRESH = 0.15   # FIX 4: flag unreliable if max basis R2 < this

results_v31 = []

for path in audio_files:
    fname = os.path.basename(path)
    stem  = os.path.splitext(fname)[0].lower()
    meta  = RECORDING_META.get(stem, {"group":"?","intensity":"?","r_frac":0.5,"skip_ms":30})
    skip  = meta.get("skip_ms", 30)   # FIX 2: per-group skip

    print(f"\n{'='*58}")
    print(f"  {fname}  [{meta['group']}/{meta['intensity']}]  skip={skip}ms")

    try:
        # FIX 2+3: adaptive pre-processing
        signal, fs, onset, win_ms, snr = preprocess_v31(path, skip_ms=skip)
        t = np.linspace(0, len(signal)/fs, len(signal), endpoint=False)
        print(f"  window={win_ms:.0f}ms  SNR={snr:.1f}dB  samples={len(signal)}")

        damping = ChendaDamping()

        # Shape identification with active-region R2
        r2s = {}
        for shape in SHAPES_LIST:
            Phi = build_basis(BASES[shape]['freqs'], BASES[shape]['alphas'], t)
            _, _, _, r2 = pgsd_decompose_ar(signal, Phi)
            r2s[shape] = round(float(r2), 4)

        max_r2 = max(r2s.values())
        best   = max(r2s, key=r2s.get)

        # FIX 4: reliability gate
        reliable = max_r2 >= R2_RELIABLE_THRESH
        if not reliable:
            print(f"  UNRELIABLE: max basis R2={max_r2:.4f} < {R2_RELIABLE_THRESH}")
            print(f"  r2_by_shape: {r2s}")
            results_v31.append(dict(
                file=fname, stem=stem, meta=meta, reliable=False,
                r2=max_r2, r2_by_shape=r2s, shape=best,
                T_est=None, f1_hz=None, r_cm=None, CI=None,
                signal=signal, fit=np.zeros_like(signal), fs=fs, t=t,
                note="Unreliable: all basis R2 below threshold"))
            continue

        print(f"  R2 by shape: {r2s}  -> {best}")

        # FIX 1: pgsd_refine with f1 clamped to [150, 280 Hz]
        fr_init = BASES[best]['freqs'].copy()
        al_init = BASES[best]['alphas'].copy()
        f1_init = float(fr_init[0])

        f_rf, al_rf, c_f, res_f, fit_f, r2h, _ = pgsd_refine(
            signal, fr_init, al_init, damping, t,
            n_iter=20, lr=0.20, search_hz=45.0, verbose=False,
            f1_min=150.0, f1_max=280.0)       # <-- FIX 1

        f1_refined = float(f_rf[0])
        T_est      = T_ANALYSIS * (f1_refined / f1_init)**2

        # Active-region R2 on final fit
        Phi_f = build_basis(f_rf, al_rf, t)
        _, _, fit_final, r2_final = pgsd_decompose_ar(signal, Phi_f)

        # Early-decay T60
        T60_ref = T60_REF_D[best]
        f_nyq = fs / 2.0
        flo = max(20., f1_refined*0.82); fhi = min(f_nyq*0.97, f1_refined*1.18)
        b_, a_ = butter(4, [flo/f_nyq, fhi/f_nyq], btype='band')
        env = np.abs(filtfilt(b_, a_, signal))
        sm  = max(1, int(fs*0.005))
        env = np.convolve(env, np.ones(sm)/sm, mode='same')
        t_a = np.arange(len(signal))/fs
        pk  = env.max()
        abo = np.where(env > pk*0.05)[0]
        i0  = abo[0] if len(abo)>10 else 0
        bl  = np.where(env[i0:] < pk/10)[0]
        i1  = i0+bl[0] if len(bl)>=3 else abo[-1] if len(abo)>10 else len(signal)//2
        if i1 > i0+5:
            ef = np.log(np.maximum(env[i0:i1], 1e-12)); fin = np.isfinite(ef)
            alm = max(-np.polyfit(t_a[i0:i1][fin], ef[fin], 1)[0], 1e-3) if fin.sum()>5                   else damping.decay_rate(2*np.pi*f1_refined)
        else:
            alm = damping.decay_rate(2*np.pi*f1_refined)
        T60_m = np.log(1000)/alm
        CI    = float(np.clip(100*(1-T60_m/T60_ref), 0, 100))

        # Position
        fr_t, mo_t, al_t, rg, thg, _ = get_spectral_basis(best, T=T_est, n_modes=N_MODES)
        WIN = min(150, len(signal)//4)
        r_s = np.linspace(0., 0.97, 80)
        obs = signal[:WIN]; obs = obs/(np.sqrt(np.sum(obs**2))+1e-12)
        corrs = []
        for rt in r_s:
            _, at = synthesise_strike(fr_t, mo_t, al_t, rg, thg,
                r_s=rt*R_PHYS, theta_s=0.0, t=t[:WIN], snr_db=999)
            Pw = build_basis(fr_t, al_t, t[:WIN])
            tm = Pw @ at; tm = tm/(np.sqrt(np.sum(tm**2))+1e-12)
            corrs.append(float(tm @ obs))
        r_est = float(r_s[int(np.argmax(corrs))])

        res_dict = dict(
            file=fname, stem=stem, meta=meta, reliable=True,
            shape=best, T_est=round(T_est,1),
            f1_hz=round(f1_refined,2), r_cm=round(r_est*R_PHYS*100,1),
            CI=round(CI,1), T60_s=round(T60_m,3),
            r2=round(r2_final,4),
            r2_by_shape={k:round(v,4) for k,v in r2s.items()},
            signal=signal, fit=fit_final, fs=fs, t=t, note="")

        v3 = V3_R2.get(stem, 0)
        delta = r2_final - v3
        print(f"  T={T_est:.0f}N/m  f1={f1_refined:.1f}Hz  r={r_est*R_PHYS*100:.1f}cm  "
              f"CI={CI:.0f}  R2={r2_final:.4f}  (v3={v3:.4f}  delta={delta:+.4f})")
        results_v31.append(res_dict)

    except Exception as e:
        import traceback; traceback.print_exc()

print(f"\n\nDone: {len(results_v31)} recordings")
good = [r for r in results_v31 if r['reliable']]
if good:
    r2s = [r['r2'] for r in good]
    print(f"Reliable: {len(good)}/{len(results_v31)}  mean_R2={np.mean(r2s):.4f}  best={max(r2s):.4f}")


  beat_1.wav  [?/?]  skip=30ms
  window=263ms  SNR=7.6dB  samples=5807
  R2 by shape: {'Cylinder': 0.3011, 'Hourglass': 0.1793, 'Oval': 0.4004, 'RealChenda': 0.2823}  -> Oval
  T=6895N/m  f1=167.4Hz  r=3.1cm  CI=79  R2=0.6892  (v3=0.0000  delta=+0.6892)

  beat_10.wav  [?/?]  skip=30ms
  window=354ms  SNR=21.4dB  samples=7815
  R2 by shape: {'Cylinder': 0.0185, 'Hourglass': 0.6405, 'Oval': 0.0298, 'RealChenda': 0.0086}  -> Hourglass
  T=4294N/m  f1=222.9Hz  r=12.1cm  CI=94  R2=0.7959  (v3=0.0000  delta=+0.7959)

  beat_11.wav  [?/?]  skip=30ms
  window=367ms  SNR=22.2dB  samples=8093
  R2 by shape: {'Cylinder': 0.0192, 'Hourglass': 0.5694, 'Oval': 0.0247, 'RealChenda': 0.0171}  -> Hourglass
  T=4211N/m  f1=220.7Hz  r=0.6cm  CI=95  R2=0.7932  (v3=0.0000  delta=+0.7932)

  beat_12.wav  [?/?]  skip=30ms
  window=365ms  SNR=19.8dB  samples=8044
  R2 by shape: {'Cylinder': 0.0166, 'Hourglass': 0.5788, 'Oval': 0.0423, 'RealChenda': 0.0092}  -> Hourglass
  T=4261N/m  f1=222.0Hz  r=12.1cm  CI

## Cell 5 - Per-recording diagnostic figure

## Cell 6 - Summary comparison table

In [5]:


V1_R2 = {"centre_high":0.0015,"centre_low":0.0083,"centre_med":0.0048,
          "edge_high":0.0036,"edge_low":0.0002,"edge_med":0.0004,
          "hand_full":0.0025,"hand_half":0.0123,"hand_small":0.0418}

for res in results_v31:
    v1  = V1_R2.get(res['stem'], 0)
    v3  = V3_R2.get(res['stem'], 0)
    v31 = res['r2']
    delta = v31 - v3
    note = res.get('note','')
    if not res['reliable']: note = "UNRELIABLE - excluded"
    T_s   = f"{res['T_est']:.0f}" if res['T_est'] else "-"
    f1_s  = f"{res['f1_hz']:.1f}" if res['f1_hz'] else "-"
    marker = " GOOD" if v31>=0.5 else " OK" if v31>=0.25 else ""
    print(f"  {res['file'][:20]:20}  {v1:.4f}  {v3:.4f}  {v31:.4f}  {delta:+.4f}    "
      f"{T_s:8}  {f1_s:7}  {note}{marker}")
reliable = [r for r in results_v31 if r['reliable']]
print()
if reliable:
    r2s = [r['r2'] for r in reliable]
    print(f"  Reliable recordings: {len(reliable)}/{len(results_v31)}")
    print(f"  Mean R2 (reliable): {np.mean(r2s):.4f}")
    print(f"  Best R2:            {max(r2s):.4f}  ({reliable[np.argmax(r2s)]['file']})")
    
  

  beat_1.wav            0.0000  0.0000  0.6892  +0.6892    6895      167.4     GOOD
  beat_10.wav           0.0000  0.0000  0.7959  +0.7959    4294      222.9     GOOD
  beat_11.wav           0.0000  0.0000  0.7932  +0.7932    4210      220.7     GOOD
  beat_12.wav           0.0000  0.0000  0.7402  +0.7402    4261      222.0     GOOD
  beat_13.wav           0.0000  0.0000  0.5793  +0.5793    4206      220.6     GOOD
  beat_14.wav           0.0000  0.0000  0.6356  +0.6356    4266      222.2     GOOD
  beat_15.wav           0.0000  0.0000  0.7959  +0.7959    4294      222.9     GOOD
  beat_16.wav           0.0000  0.0000  0.7959  +0.7959    4294      222.9     GOOD
  beat_17.wav           0.0000  0.0000  0.7959  +0.7959    4294      222.9     GOOD
  beat_18.wav           0.0000  0.0000  0.6356  +0.6356    4266      222.2     GOOD
  beat_19.wav           0.0000  0.0000  0.8152  +0.8152    3394      198.2     GOOD
  beat_2.wav            0.0000  0.0000  0.6696  +0.6696    3586      203.7  

## Cell 7 - JSON output for paper

In [6]:
import os
import numpy as np
from scipy.io import wavfile

OUT_DIR_PGSD = f"{WORK}/reconstructed_beats/pgsd"
os.makedirs(OUT_DIR_PGSD, exist_ok=True)

for res in results_v31:
    if not res['reliable']:
        print(f"Skipping (unreliable): {res['file']}")
        continue

    fit  = np.array(res['fit'], dtype=np.float64)
    fs   = int(res['fs'])
    stem = os.path.splitext(res['file'])[0]

    # Normalise to [-1, 1] then convert to int16
    peak = np.abs(fit).max()
    fit_norm = fit / peak if peak > 0 else fit

    wav_data = (fit_norm * 32767).astype(np.int16)
    out_path = os.path.join(OUT_DIR_PGSD, f"{stem}_pgsd_recon.wav")
    wavfile.write(out_path, fs, wav_data)
    print(f"Saved: {out_path}  (R²={res['r2']:.4f})")

print(f"\n✅ PGSD reconstructions saved to {OUT_DIR_PGSD}")

Saved: /kaggle/working/reconstructed_beats/pgsd/beat_1_pgsd_recon.wav  (R²=0.6892)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_10_pgsd_recon.wav  (R²=0.7959)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_11_pgsd_recon.wav  (R²=0.7932)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_12_pgsd_recon.wav  (R²=0.7402)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_13_pgsd_recon.wav  (R²=0.5793)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_14_pgsd_recon.wav  (R²=0.6356)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_15_pgsd_recon.wav  (R²=0.7959)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_16_pgsd_recon.wav  (R²=0.7959)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_17_pgsd_recon.wav  (R²=0.7959)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_18_pgsd_recon.wav  (R²=0.6356)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_19_pgsd_recon.wav  (R²=0.8152)
Saved: /kaggle/working/reconstructed_beats/pgsd/beat_2_pgsd_recon.wav  (R²=0.

@fft

In [14]:
import numpy as np

def fft_reconstruct(x, sr, n_peaks=30):
    """
    Reconstruct signal by keeping only the top n_peaks
    frequency components from the FFT spectrum.
    """
    N = len(x)
    
    # Forward FFT
    X = np.fft.rfft(x)
    
    # Get magnitude spectrum
    magnitude = np.abs(X)
    
    # Find top n_peaks frequency bins by magnitude
    top_indices = np.argsort(magnitude)[-n_peaks:]
    
    # Zero out everything except top peaks
    X_filtered = np.zeros_like(X)
    X_filtered[top_indices] = X[top_indices]
    
    # Inverse FFT to reconstruct
    x_recon = np.fft.irfft(X_filtered, n=N)
    
    return x_recon


def compute_r2(x, x_hat):
    """R² between original and reconstructed signal."""
    eps = 1e-8
    return 1 - (np.sum((x - x_hat) ** 2) / (np.sum(x ** 2) + eps))


print("✅ fft_reconstruct and compute_r2 defined")

✅ fft_reconstruct and compute_r2 defined


In [15]:
import os
import numpy as np
from scipy.io import wavfile

results = []

folder = BEATS_DIR  # change if needed

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    print("Processing:", file)

    if not os.path.exists(path):
        print("❌ Missing:", file)
        continue

    # Load audio
    sr, x = wavfile.read(path)
    x = x.astype(float)

    # Normalize
    if np.max(np.abs(x)) == 0:
        print("⚠️ Silent file:", file)
        continue
    x = x / np.max(np.abs(x))

    # FFT reconstruction
    x_fft = fft_reconstruct(x, sr, n_peaks=30)  # 30 is a good value

    # R²
    r2_fft = compute_r2(x, x_fft)

    results.append({
        "file": file,
        "r2_fft": float(r2_fft)   # convert from np.float64 → float
    })

print("\n✅ DONE")
print("Total processed:", len(results))

Processing: beat_1.wav
Processing: beat_2.wav
Processing: beat_3.wav
Processing: beat_4.wav
Processing: beat_5.wav
Processing: beat_6.wav
Processing: beat_7.wav
Processing: beat_8.wav
Processing: beat_9.wav
Processing: beat_10.wav
Processing: beat_11.wav
Processing: beat_12.wav
Processing: beat_13.wav
Processing: beat_14.wav
Processing: beat_15.wav
Processing: beat_16.wav
Processing: beat_17.wav
Processing: beat_18.wav
Processing: beat_19.wav
Processing: beat_20.wav
Processing: beat_21.wav
Processing: beat_22.wav
Processing: beat_23.wav
Processing: beat_24.wav
Processing: beat_25.wav
Processing: beat_26.wav
Processing: beat_27.wav
Processing: beat_28.wav
Processing: beat_29.wav
Processing: beat_30.wav
Processing: beat_31.wav
Processing: beat_32.wav
Processing: beat_33.wav
Processing: beat_34.wav
Processing: beat_35.wav
Processing: beat_36.wav
Processing: beat_37.wav
Processing: beat_38.wav
Processing: beat_39.wav
Processing: beat_40.wav
Processing: beat_41.wav
Processing: beat_42.wav
P

In [16]:
import numpy as np

mean_fft = np.mean([r["r2_fft"] for r in results])
print("Mean FFT R2:", mean_fft)

Mean FFT R2: 0.49845600880461144


In [17]:
import os
import numpy as np
from scipy.io import wavfile

OUT_DIR_FFT = f"{WORK}/reconstructed_beats/fft"
os.makedirs(OUT_DIR_FFT, exist_ok=True)

folder = BEATS_DIR

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("❌ Missing:", file)
        continue

    sr, x = wavfile.read(path)
    x = x.astype(float)

    if np.max(np.abs(x)) == 0:
        print("⚠️ Silent:", file)
        continue

    x = x / np.max(np.abs(x))

    # Reconstruct
    x_fft = fft_reconstruct(x, sr, n_peaks=30)

    # Normalise and save
    peak = np.abs(x_fft).max()
    x_fft_norm = x_fft / peak if peak > 0 else x_fft

    wav_data = (x_fft_norm * 32767).astype(np.int16)
    out_path = os.path.join(OUT_DIR_FFT, f"beat_{i}_fft_recon.wav")
    wavfile.write(out_path, sr, wav_data)
    print(f"Saved: beat_{i}  R²={compute_r2(x, x_fft):.4f}")

print(f"\n✅ FFT reconstructions saved to {OUT_DIR_FFT}")

Saved: beat_1  R²=0.5402
Saved: beat_2  R²=0.2956
Saved: beat_3  R²=0.5005
Saved: beat_4  R²=0.5556
Saved: beat_5  R²=0.4099
Saved: beat_6  R²=0.6117
Saved: beat_7  R²=0.6325
Saved: beat_8  R²=0.6117
Saved: beat_9  R²=0.5396
Saved: beat_10  R²=0.5261
Saved: beat_11  R²=0.5301
Saved: beat_12  R²=0.5209
Saved: beat_13  R²=0.5050
Saved: beat_14  R²=0.4776
Saved: beat_15  R²=0.5261
Saved: beat_16  R²=0.5261
Saved: beat_17  R²=0.5261
Saved: beat_18  R²=0.4776
Saved: beat_19  R²=0.4099
Saved: beat_20  R²=0.4163
Saved: beat_21  R²=0.4051
Saved: beat_22  R²=0.3946
Saved: beat_23  R²=0.3935
Saved: beat_24  R²=0.3946
Saved: beat_25  R²=0.2956
Saved: beat_26  R²=0.3678
Saved: beat_27  R²=0.4099
Saved: beat_28  R²=0.2636
Saved: beat_29  R²=0.4776
Saved: beat_30  R²=0.5245
Saved: beat_31  R²=0.5005
Saved: beat_32  R²=0.4104
Saved: beat_33  R²=0.5261
Saved: beat_34  R²=0.6060
Saved: beat_35  R²=0.7030
Saved: beat_36  R²=0.5390
Saved: beat_37  R²=0.4534
Saved: beat_38  R²=0.6575
Saved: beat_39  R²=0.

sinusoidal

In [10]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import stft, istft, find_peaks

# -------------------------------
# R² function
# -------------------------------
def compute_r2(x, x_hat):
    return 1 - (np.sum((x - x_hat)**2) / np.sum(x**2))


# -------------------------------
# Sinusoidal Model
# -------------------------------
def sinusoidal_model(x, sr, n_peaks=4):
    f, t, Zxx = stft(x, fs=sr, nperseg=1024, noverlap=512)

    magnitude = np.abs(Zxx)
    phase = np.angle(Zxx)

    Zxx_hat = np.zeros_like(Zxx, dtype=complex)

    for i in range(Zxx.shape[1]):
        mag_frame = magnitude[:, i]

        # stricter peak selection
        peaks, _ = find_peaks(mag_frame, height=np.max(mag_frame)*0.15)

        peaks = sorted(peaks, key=lambda p: mag_frame[p], reverse=True)
        peaks = peaks[:n_peaks]

        for p in peaks:
            Zxx_hat[p, i] = mag_frame[p] * np.exp(1j * phase[p, i])

    _, x_hat = istft(Zxx_hat, fs=sr)
    x_hat = x_hat[:len(x)]

    return x_hat


# -------------------------------
# MAIN LOOP (50 beats)
# -------------------------------
folder = BEATS_DIR

r2_values = []

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("Missing:", file)
        continue

    sr, x = wavfile.read(path)
    x = x.astype(float)

    if np.max(np.abs(x)) == 0:
        continue

    x = x / np.max(np.abs(x))

    x_hat = sinusoidal_model(x, sr, n_peaks=10)

    r2 = compute_r2(x, x_hat)
    r2_values.append(r2)

# -------------------------------
# FINAL OUTPUT
# -------------------------------
mean_r2 = np.mean(r2_values)

print("Mean Sinusoidal R2:", mean_r2)

Mean Sinusoidal R2: 0.5942925015292164


In [11]:
import os
import numpy as np
from scipy.io import wavfile

OUT_DIR_SIN = f"{WORK}/reconstructed_beats/sinusoidal"
os.makedirs(OUT_DIR_SIN, exist_ok=True)

folder = BEATS_DIR

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("❌ Missing:", file)
        continue

    sr, x = wavfile.read(path)
    x = x.astype(float)

    if np.max(np.abs(x)) == 0:
        print("⚠️ Silent:", file)
        continue

    x = x / np.max(np.abs(x))

    # Reconstruct
    x_hat = sinusoidal_model(x, sr, n_peaks=10)

    # Normalise and save
    peak = np.abs(x_hat).max()
    x_hat_norm = x_hat / peak if peak > 0 else x_hat

    wav_data = (x_hat_norm * 32767).astype(np.int16)
    out_path = os.path.join(OUT_DIR_SIN, f"beat_{i}_sinusoidal_recon.wav")
    wavfile.write(out_path, sr, wav_data)
    print(f"Saved: beat_{i}  R²={compute_r2(x, x_hat):.4f}")

print(f"\n✅ Sinusoidal reconstructions saved to {OUT_DIR_SIN}")

Saved: beat_1  R²=0.6224
Saved: beat_2  R²=0.5537
Saved: beat_3  R²=0.5936
Saved: beat_4  R²=0.5392
Saved: beat_5  R²=0.6347
Saved: beat_6  R²=0.6168
Saved: beat_7  R²=0.6609
Saved: beat_8  R²=0.6168
Saved: beat_9  R²=0.5079
Saved: beat_10  R²=0.5042
Saved: beat_11  R²=0.5210
Saved: beat_12  R²=0.5193
Saved: beat_13  R²=0.5172
Saved: beat_14  R²=0.6239
Saved: beat_15  R²=0.5042
Saved: beat_16  R²=0.5042
Saved: beat_17  R²=0.5042
Saved: beat_18  R²=0.6239
Saved: beat_19  R²=0.6347
Saved: beat_20  R²=0.6372
Saved: beat_21  R²=0.6511
Saved: beat_22  R²=0.6529
Saved: beat_23  R²=0.6648
Saved: beat_24  R²=0.6529
Saved: beat_25  R²=0.5537
Saved: beat_26  R²=0.6221
Saved: beat_27  R²=0.6347
Saved: beat_28  R²=0.5496
Saved: beat_29  R²=0.6239
Saved: beat_30  R²=0.6494
Saved: beat_31  R²=0.5936
Saved: beat_32  R²=0.5927
Saved: beat_33  R²=0.5042
Saved: beat_34  R²=0.6219
Saved: beat_35  R²=0.6811
Saved: beat_36  R²=0.5351
Saved: beat_37  R²=0.6578
Saved: beat_38  R²=0.6767
Saved: beat_39  R²=0.

NMF

In [12]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import stft, istft
from sklearn.decomposition import NMF

# -------------------------------
# GLOBAL STFT SETTINGS
# -------------------------------
NPERSEG = 1024
NOVERLAP = 512
WINDOW = 'hann'


# -------------------------------
# R² function
# -------------------------------
def compute_r2(x, x_hat):
    eps = 1e-8
    return 1 - (np.sum((x - x_hat)**2) / (np.sum(x**2) + eps))


# -------------------------------
# Spectral error
# -------------------------------
def spectral_error(V, V_hat):
    return np.linalg.norm(V - V_hat) / (np.linalg.norm(V) + 1e-8)


# -------------------------------
# ALIGN SIGNALS (CRITICAL FIX)
# -------------------------------
def align_signals(x, x_hat):
    corr = np.correlate(x_hat, x, mode='full')
    shift = np.argmax(corr) - len(x) + 1

    if shift > 0:
        x_hat = x_hat[shift:]
        x = x[:len(x_hat)]
    else:
        x = x[-shift:]
        x_hat = x_hat[:len(x)]

    min_len = min(len(x), len(x_hat))
    return x[:min_len], x_hat[:min_len]


# -------------------------------
# GRIFFIN-LIM (STABLE)
# -------------------------------
def griffin_lim(V, sr, n_iter=100):
    angles = np.exp(1j * 2 * np.pi * np.random.rand(*V.shape))
    Zxx = V * angles

    for _ in range(n_iter):
        _, x_temp = istft(
            Zxx,
            fs=sr,
            window=WINDOW,
            nperseg=NPERSEG,
            noverlap=NOVERLAP
        )

        _, _, Zxx_temp = stft(
            x_temp,
            fs=sr,
            window=WINDOW,
            nperseg=NPERSEG,
            noverlap=NOVERLAP
        )

        Zxx = V * np.exp(1j * np.angle(Zxx_temp))

    _, x_recon = istft(
        Zxx,
        fs=sr,
        window=WINDOW,
        nperseg=NPERSEG,
        noverlap=NOVERLAP
    )

    return x_recon


# -------------------------------
# FAIR NMF MODEL
# -------------------------------
def nmf_model_fair(x, sr, n_components=3):
    _, _, Zxx = stft(
        x,
        fs=sr,
        window=WINDOW,
        nperseg=NPERSEG,
        noverlap=NOVERLAP
    )

    V = np.abs(Zxx)

    model = NMF(
        n_components=n_components,
        init='random',
        max_iter=300,
        random_state=0
    )

    W = model.fit_transform(V)
    H = model.components_

    V_hat = np.dot(W, H)

    x_hat = griffin_lim(V_hat, sr, n_iter=100)

    return x_hat, V, V_hat


# -------------------------------
# MAIN LOOP
# -------------------------------
folder = BEATS_DIR

r2_values = []
spec_errors = []

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("Missing:", file)
        continue

    sr, x = wavfile.read(path)
    x = x.astype(float)

    if np.max(np.abs(x)) == 0:
        continue

    # normalize input
    x = x / (np.max(np.abs(x)) + 1e-8)

    # run model
    x_hat, V, V_hat = nmf_model_fair(x, sr, n_components=3)

    # ALIGN SIGNALS (KEY STEP)
    x, x_hat = align_signals(x, x_hat)

    # normalize AFTER alignment
    x = x / (np.max(np.abs(x)) + 1e-8)
    x_hat = x_hat / (np.max(np.abs(x_hat)) + 1e-8)

    # metrics
    r2 = compute_r2(x, x_hat)
    spec_err = spectral_error(V, V_hat)

    r2_values.append(r2)
    spec_errors.append(spec_err)


# -------------------------------
# FINAL OUTPUT
# -------------------------------
mean_r2 = np.mean(r2_values)
mean_spec_err = np.mean(spec_errors)

print("FAIR NMF Results:")
print("Mean R2:", mean_r2)
print("Mean Spectral Error:", mean_spec_err)

FAIR NMF Results:
Mean R2: -0.2224654518469656
Mean Spectral Error: 0.163564212761095


In [13]:
import os
import numpy as np
from scipy.io import wavfile

OUT_DIR_NMF = f"{WORK}/reconstructed_beats/nmf"
os.makedirs(OUT_DIR_NMF, exist_ok=True)

folder = BEATS_DIR

for i in range(1, 51):
    file = f"beat_{i}.wav"
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("❌ Missing:", file)
        continue

    sr, x = wavfile.read(path)
    x = x.astype(float)

    if np.max(np.abs(x)) == 0:
        print("⚠️ Silent:", file)
        continue

    x = x / (np.max(np.abs(x)) + 1e-8)

    # Reconstruct
    x_hat, V, V_hat = nmf_model_fair(x, sr, n_components=3)

    # Align signals (same as your original loop)
    x_a, x_hat_a = align_signals(x, x_hat)
    x_a     = x_a     / (np.max(np.abs(x_a))     + 1e-8)
    x_hat_a = x_hat_a / (np.max(np.abs(x_hat_a)) + 1e-8)

    # Save
    wav_data = (x_hat_a * 32767).astype(np.int16)
    out_path = os.path.join(OUT_DIR_NMF, f"beat_{i}_nmf_recon.wav")
    wavfile.write(out_path, sr, wav_data)
    print(f"Saved: beat_{i}  R²={compute_r2(x_a, x_hat_a):.4f}  SpecErr={spectral_error(V, V_hat):.4f}")

print(f"\n✅ NMF reconstructions saved to {OUT_DIR_NMF}")

Saved: beat_1  R²=0.0245  SpecErr=0.1903
Saved: beat_2  R²=-0.3467  SpecErr=0.1899
Saved: beat_3  R²=-0.1191  SpecErr=0.1865
Saved: beat_4  R²=-0.0554  SpecErr=0.1195
Saved: beat_5  R²=-0.2453  SpecErr=0.1987
Saved: beat_6  R²=0.3195  SpecErr=0.1338
Saved: beat_7  R²=0.2744  SpecErr=0.1390
Saved: beat_8  R²=0.3481  SpecErr=0.1338
Saved: beat_9  R²=0.0703  SpecErr=0.1231
Saved: beat_10  R²=0.2076  SpecErr=0.1156
Saved: beat_11  R²=0.4530  SpecErr=0.1196
Saved: beat_12  R²=0.3902  SpecErr=0.1201
Saved: beat_13  R²=-0.3188  SpecErr=0.1199
Saved: beat_14  R²=-0.9952  SpecErr=0.2154
Saved: beat_15  R²=-0.2871  SpecErr=0.1156
Saved: beat_16  R²=0.0313  SpecErr=0.1156
Saved: beat_17  R²=0.3566  SpecErr=0.1156
Saved: beat_18  R²=-1.4983  SpecErr=0.2154
Saved: beat_19  R²=-0.4423  SpecErr=0.1987
Saved: beat_20  R²=-0.1036  SpecErr=0.2129
Saved: beat_21  R²=-0.5201  SpecErr=0.2106
Saved: beat_22  R²=-0.1805  SpecErr=0.1940
Saved: beat_23  R²=-0.2801  SpecErr=0.1790
Saved: beat_24  R²=-0.4869  Sp

Sinusoidal R2: 0.6318326598104433
